# All-23 RGB-Geodesic Cascade
Hucreleri sirasiyla calistirin. Once smoke test, ardindan A100 veya 5-fold CV kosusu yapin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import subprocess

CODE_ROOT = Path('/content/comparative-study')
REPO_URL = 'https://github.com/eckdev/comparative-study.git'
if CODE_ROOT.exists():
    subprocess.run(['git', '-C', str(CODE_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(CODE_ROOT)], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(CODE_ROOT / 'all23_rgb_geodesic_cascade/requirements.txt')], check=True)
print('CODE_ROOT:', CODE_ROOT)

In [ ]:
paths = {
    'dataset': Path('/content/drive/MyDrive/orthodontic/data/dataset'),
    'split': CODE_ROOT / 'shared_splits/orthodontic_180_60_60_seed42.json',
    'legacy_transform': Path('/content/drive/MyDrive/orthodontic/transforms/orthodontic_procrustes_rigid_20260627_143801'),
    'agh_v6': Path('/content/drive/MyDrive/orthodontic/diffusion_runs/aghformer_v6_stage2_raw_fine_refiner_p12000'),
    'stacker': Path('/content/drive/MyDrive/orthodontic/diffusion_runs/shape_prior_stacker'),
}
for name, path in paths.items():
    print(f'{name:18s}', path.exists(), path)
assert paths['dataset'].exists(), 'Drive dataset bulunamadi'

## 1. Smoke test

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset smoke

## 2A. A100 sabit split ana kosu

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset a100 --seed 42

## 2B. Leakage-free Stage 1 + ROI preflight
Bu hucre Stage 2'yi baslatmadan nested OOF Stage 1 modellerini egitir, cache'ler ve tum foldlarin ROI kapsamlarini denetler. Ilk calisma uzun surebilir.

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset cv_preflight --seed 42

## 2C. Leakage-free 5-fold yayin kosusu
Preflight bes fold icin basarili olduktan sonra calistirin; Stage 1 checkpoint ve cache'leri yeniden kullanilir.

In [ ]:
import csv
cv_run = Path('/content/drive/MyDrive/orthodontic/all23_rgb_geodesic_runs/publication_cv_stage1_v4_seed42')
preflight = cv_run / 'preflight_oracle_summary.csv'
if preflight.exists():
    for row in csv.DictReader(preflight.open()):
        print('fold', row['fold'], 'stage1_val', row['stage1_validation_ale'], 'oracle', row['validation_oracle_ale'], 'passed', row['passed'])
else:
    print('Preflight raporu henuz yok:', preflight)

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset cv --seed 42

In [ ]:
import json
result = cv_run / 'summary_metrics.json'
if result.exists():
    metrics = json.loads(result.read_text())
    print('Fold ALE mean:', metrics['fold_ale_mean'])
    print('Fold ALE std:', metrics['fold_ale_std'])
    for fold in metrics['folds']:
        print('fold', fold['fold'], 'ALE', fold['ale'], 'Core20', fold['core20_ale'], 'Hard3', fold['hard3_ale'])
else:
    print('Henuz sonuc dosyasi yok:', result)

## 3. E9 Hard3 candidate-ranker gelistirme kosusu
Bu kosu eski v4 alignment, Stage 1 ve ROI cache'lerini yeniden kullanir. Once yalniz Fold 1 calisir; sonuc kabul kapilarini gecmeden tum foldlari baslatmayin.

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset e9_dev --seed 42

In [ ]:
import json
e9_dev = Path('/content/drive/MyDrive/orthodontic/all23_rgb_geodesic_runs/publication_e9_cv_seed42/fold_1')
metrics_path = e9_dev / 'metrics_val.json'
if metrics_path.exists():
    m = json.loads(metrics_path.read_text())
    print('E9 Fold 1 validation:', m['overall']['ale'])
    print('Core20:', m['core20']['ale'], 'Hard3:', m['hard3']['ale'])
    print('p95:', m['overall']['p95'], 'max:', m['overall']['max'])
else:
    print('E9 gelistirme sonucu henuz yok:', metrics_path)

In [ ]:
!python -u evaluate_e9_gate.py \
  --baseline-root /content/drive/MyDrive/orthodontic/all23_rgb_geodesic_runs/publication_cv_stage1_v4_seed42 \
  --candidate-root /content/drive/MyDrive/orthodontic/all23_rgb_geodesic_runs/publication_e9_cv_seed42 \
  --fold 1

## 4. E9 leakage-free 5-fold kosusu
Yalniz Fold 1 validation kabul kapilarini gectiyse calistirin. Tamamlanan foldlar yeniden baslatmada otomatik atlanir; yarim kalan fold last_model.pth checkpointinden devam eder.

In [ ]:
%cd /content/comparative-study/all23_rgb_geodesic_cascade
!python -u colab_run_all23_rgb_geodesic.py --preset e9_cv --seed 42